In [ ]:
#!/usr/bin/env python3
"""
KAGGLE JOB — THE DECISIVE INTERVENTIONAL TEST for the causal-router thesis.
Brian, 2026-07-18.  Track: docs/TASK_causal_router_prototype.md, STATE.md "CAUSAL-ROUTER track".

THE QUESTION
------------
Does CAUSAL trans-eQTL wiring predict PERTURBATION RESPONSE better than
OBSERVATIONAL co-expression grouping?

WHY THIS AND NOT WHAT WE ALREADY RAN
------------------------------------
In-sandbox we showed (fair, non-circular, contexts-A-build / contexts-B-eval):
    random gene sets      +0.133
    causal trans loci     +0.304   (z=+17.6 vs random -> wiring carries REAL info)
    co-expression groups  +0.461   (causal LOSES, z=-16.0)
But that is an OBSERVATIONAL benchmark, where co-expression wins by construction
(it *is* correlation structure). The entire theoretical case for causal instruments
is INTERVENTIONAL: under perturbation, correlation is known to fail. This job runs
that test. It is the single experiment that decides the router thesis.

PRE-REGISTERED DECISION RULE (fixed before running; do not edit after seeing output)
-----------------------------------------------------------------------------------
Primary metric: mean Spearman r between a held-out gene's true perturbation
log2FC-profile and the response profile estimated from its group's HELD-IN genes.
  PASS   : causal beats co-expression by z > 3 AND beats random by z > 3
           -> causal instruments add value exactly where correlation fails. Build the router.
  NULL   : causal ~ co-expression (|z| <= 3) -> causal wiring adds nothing beyond
           correlation even under intervention. Router thesis dies. This is a REAL,
           publishable finding about the field's proposed "add causality" fix.
  LOSS   : co-expression beats causal by z > 3 -> same conclusion as NULL, stated harder.
Report the number either way. A null here is a result, not a failure.

FAIRNESS CONSTRAINTS (non-negotiable, this is what makes the test mean anything)
-------------------------------------------------------------------------------
1. Co-expression groups are built ONLY from BASELINE (unperturbed) expression.
   They never see the response data they are evaluated on.
2. All three arms are SIZE-MATCHED per group.
3. Held-in / held-out split within every group -> no gene predicts itself.
4. Multiple seeds; report mean and dispersion, not one lucky split.

INPUTS (attach as Kaggle datasets; script auto-discovers)
--------------------------------------------------------
  eqtlgen_trans_sig_20180904.txt.gz     - causal wiring (SNPChr,SNPPos,Zscore,Gene)
  g1_cis_trans_joint_table.csv          - baseline expression per gene x cell-type bucket
  scbloodnl_de_*.parquet                - PERTURBATION response (stimulation log2FC per gene)
                                          [Kaggle has pyarrow; the sandbox does not]
Optional larger arm: any Perturb-seq / Virtual Cell Challenge response matrix in the
same shape (rows=genes, cols=perturbations, values=log2FC) -> set RESPONSE_GLOB.
"""
import os, glob, sys
import numpy as np, pandas as pd

SEEDS       = [0, 1, 2, 3, 4]
MIN_GROUP   = 20           # min genes per group to be usable
# Prefer the per-condition DE (v2, ~24 response columns). Falls back to the
# pooled v1 files (4 columns), which will almost certainly return WITHHELD.
RESPONSE_GLOB = "scbloodnl_de_percond_*.parquet"
RESPONSE_GLOB_FALLBACK = "scbloodnl_de_*.parquet"
SEARCH_ROOTS  = ["/kaggle/input", "./data/processed", "./data/raw", "."]

def find(pattern):
    for root in SEARCH_ROOTS:
        hits = glob.glob(os.path.join(root, "**", pattern), recursive=True)
        # Add this list comprehension to filter out directories
        hits = [h for h in hits if os.path.isfile(h)]
        if hits: return sorted(hits)
    return []

def spearman(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    m = ~(np.isnan(x) | np.isnan(y)); x, y = x[m], y[m]
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0: return np.nan
    return np.corrcoef(pd.Series(x).rank(), pd.Series(y).rank())[0, 1]

def zdiff(a, b):
    a = np.asarray(a); b = np.asarray(b)
    a = a[~np.isnan(a)]; b = b[~np.isnan(b)]
    if len(a) < 2 or len(b) < 2: return nap.nan
    return (a.mean() - b.mean()) / np.sqrt(a.var()/len(a) + b.var()/len(b))

# ---------------------------------------------------------------- load
print("=== LOADING ===")
# Change the search string to match the extracted file's actual name
w = find("*trans-eQTLsFDR0.05*.txt")
if not w: sys.exit("FATAL: trans-eQTL wiring file not found.")
te = pd.read_csv(w[0], sep="\t", usecols=["SNPChr","SNPPos","Zscore","Gene"])
te["locus"] = te.SNPChr.astype(str) + ":" + (te.SNPPos // 1_000_000).astype(int).astype(str)
print(f"wiring: {len(te):,} edges | {te.locus.nunique()} loci | {te.Gene.nunique()} targets")

b = find("g1_cis_trans_joint_table.csv")
if not b: sys.exit("FATAL: baseline expression table not found.")
j = pd.read_csv(b[0], usecols=["gene_id","bucket","mean_expr"])
BASE = j.pivot_table(index="gene_id", columns="bucket", values="mean_expr", aggfunc="mean")
bc = list(BASE.columns)
Bz = BASE[bc].sub(BASE[bc].mean(axis=1), axis=0)
Bz = Bz.div(BASE[bc].std(axis=1).replace(0, np.nan), axis=0).dropna()
print(f"baseline expression: {Bz.shape[0]} genes x {Bz.shape[1]} contexts (co-expression built ONLY from this)")

rfiles = find(RESPONSE_GLOB)
if rfiles:
    print(f"using PER-CONDITION response files (v2): {len(rfiles)}")
else:
    rfiles = [f for f in find(RESPONSE_GLOB_FALLBACK) if "percond" not in f]
    print("WARNING: per-condition (v2) DE files not found; falling back to POOLED v1 files. "
          "v1 pools all 6 stim conditions into one binary -> ~4 response columns -> the "
          "verdict will be WITHHELD. Run kaggle_1m_scbloodnl_de_percondition.py first.")
if not rfiles: sys.exit("FATAL: no perturbation-response files found. This job is meaningless without them.")
resp = {}
for f in rfiles:
    try:
        d = pd.read_parquet(f)
    except Exception as e:
        print(f"  skip {f}: {e}"); continue
    gcol = next((c for c in ["gene","gene_id","names","index"] if c in d.columns), None)
    # VERIFIED schema of our scbloodnl_de_*.parquet (decoded from the parquet footer):
    # ['group','gene','n_donors_used','baseline_mean','mean_diff_stim_minus_baseline','p','q']
    fcol = next((c for c in ["mean_diff_stim_minus_baseline","log2FC","logFC",
                             "log2FoldChange","lfc","coef"] if c in d.columns), None)
    if gcol is None or fcol is None:
        print(f"  skip {f}: need a gene column and an effect column; saw {list(d.columns)}"); continue
    # Split on every present key column so each (cell type x condition) becomes a
    # SEPARATE response column -- that is where profile power comes from.
    # v1 files have only 'group' (= cell type, stim pooled) -> 4 columns total.
    # v2 percond files add 'condition' -> up to 24 columns. Prefer v2.
    keys = [c for c in ["group", "condition"] if c in d.columns]
    if keys:
        for kv, sub in d.groupby(keys):
            tag = kv if isinstance(kv, str) else "|".join(map(str, kv))
            resp[tag] = sub.set_index(gcol)[fcol]
    else:
        resp[os.path.basename(f)] = d.set_index(gcol)[fcol]
if not resp: sys.exit("FATAL: response files found but none had usable gene/log2FC columns.")
R = pd.DataFrame(resp).dropna(how="all")
R = R.loc[R.index.intersection(Bz.index)]
print(f"RESPONSE matrix: {R.shape[0]} genes x {R.shape[1]} perturbation conditions -> {list(R.columns)}")
if R.shape[1] < 2:
    print("WARNING: <2 perturbation conditions. Profile correlation is degenerate; "
          "verdict will be withheld. Attach more conditions.")

# ---------------------------------------------------------------- groups
usable = set(R.index) & set(Bz.index)
loci = {l: [g for g in grp.Gene.unique() if g in usable] for l, grp in te.groupby("locus")}
loci = {l: g for l, g in loci.items() if len(g) >= MIN_GROUP}
print(f"\ncausal loci usable (>= {MIN_GROUP} targets with response+baseline): {len(loci)}")
if len(loci) < 10: print("WARNING: few usable loci -> low power. Interpret cautiously.")

idx = np.array(sorted(usable))
Bn = Bz.loc[idx].values
Bn = Bn / (np.linalg.norm(Bn, axis=1, keepdims=True) + 1e-9)   # for co-expression neighbours

def heldout_scores(genes, rng):
    """Estimate group response from held-IN genes; score prediction of held-OUT genes."""
    g = np.array([x for x in genes if x in R.index]); rng.shuffle(g)
    if len(g) < 10: return []
    h = len(g)//2; P, Q = g[:h], g[h:]
    act = R.loc[P].mean(axis=0).values           # group's perturbation-response profile
    out = []
    for t in Q:
        out.append(spearman(act, R.loc[t].values))
    return [v for v in out if not np.isnan(v)]

# ---------------------------------------------------------------- run
print("\n=== RUNNING (causal | co-expression | random), size-matched, held-out ===")
res = {"causal": [], "coexpr": [], "random": []}
for seed in SEEDS:
    rng = np.random.default_rng(seed)
    for l, gs in loci.items():
        n = len(gs)
        res["causal"] += heldout_scores(gs, rng)
        s = rng.integers(len(idx))                       # co-expression group: seed + n-1 nearest
        sims = Bn @ Bn[s]                                # built ONLY on baseline expression
        res["coexpr"] += heldout_scores(list(idx[np.argsort(-sims)[:n]]), rng)
        res["random"] += heldout_scores(list(rng.choice(idx, size=n, replace=False)), rng)

print("\n=== RESULT: predicting HELD-OUT genes' perturbation response ===")
for k in ["random", "causal", "coexpr"]:
    a = np.array(res[k]); a = a[~np.isnan(a)]
    print(f"  {k:8s}: mean r = {a.mean():+.4f}   (n = {len(a):,} held-out genes)")
z_cr = zdiff(res["causal"], res["random"])
z_cc = zdiff(res["causal"], res["coexpr"])
print(f"\n  causal vs random        z = {z_cr:+.1f}")
print(f"  causal vs co-expression z = {z_cc:+.1f}")

print("\n=== PRE-REGISTERED VERDICT ===")
if R.shape[1] < 2:
    print("  WITHHELD - too few perturbation conditions for a profile correlation.")
elif np.isnan(z_cc) or np.isnan(z_cr):
    print("  WITHHELD - insufficient data for a stable comparison.")
elif z_cc > 3 and z_cr > 3:
    print("  PASS  - causal wiring beats co-expression AND random under intervention.")
    print("          Causal instruments add value exactly where correlation fails. BUILD THE ROUTER.")
elif z_cc < -3:
    print("  LOSS  - co-expression beats causal wiring even under intervention.")
    print("          The router thesis dies. Report it: 'adding causal genetics does not")
    print("          beat correlation for perturbation prediction' is a real finding about")
    print("          the field's proposed fix.")
else:
    print("  NULL  - causal ~ co-expression. Same conclusion as LOSS, stated softer.")
    print("          Router thesis does not survive. Publish the null.")
print("\nDone. Copy the full output back into STATE.md verbatim - including a null.")
